Simple Neural Network Model


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residue3DPredictor(nn.Module):
    def __init__(self, window_size=20, num_classes=4):  # window_size changed to 10
        super().__init__()
        self.window_size = window_size

        # First convolution layer (4 channels → 32 channels)
        self.initial_conv = nn.Conv1d(num_classes, 32, kernel_size=9, padding=8)  # padding changed to 8

        # 10 convolutional layers (32 → 32), with kernel_size=9, padding=8
        self.middle_convs = nn.Sequential(
            *[nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=9, padding=8),  # padding changed to 8
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(0.1)
            ) for _ in range(10)]
        )

        # Final convolution (32 → 128)
        self.final_conv = nn.Conv1d(32, 128, kernel_size=9, padding=8)  # padding changed to 8

        # Pool to fixed window size
        self.pool = nn.AdaptiveAvgPool1d(output_size=window_size)  # output size changed to 10

        # Fully connected to 3D coordinates
        self.fc = nn.Linear(128 * window_size, 3)  # adjusts automatically to window_size=10

    def forward(self, x):
        x = F.relu(self.initial_conv(x))     # [B, 32, W]
        x = self.middle_convs(x)             # [B, 32, W]
        x = F.relu(self.final_conv(x))       # [B, 128, W]
        x = self.pool(x)                     # [B, 128, 10]
        x = x.view(x.size(0), -1)            # [B, 128 * 10]
        return self.fc(x)


Dataset Loader

In [2]:
class RNADataset(torch.utils.data.Dataset):
    def __init__(self, df, window_size=20):
        self.window_size = window_size
        self.data = []
        self.labels = []
        seq = df['resname'].values
        coords = df[['x_1', 'y_1', 'z_1']].values

        mapping = {'A': [1,0,0,0], 'U': [0,1,0,0], 'C': [0,0,1,0], 'G': [0,0,0,1]}
        pad = [0, 0, 0, 0]

        for i in range(len(seq)):
            context = []
            for j in range(i - window_size//2, i + window_size//2 + 1):
                if 0 <= j < len(seq):
                    context.append(mapping[seq[j]])
                else:
                    context.append(pad)
            self.data.append(torch.tensor(context).T)  # shape: [4, window_size]
            self.labels.append(torch.tensor(coords[i]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx].float(), self.labels[idx].float()


Training the Model

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Load the data
df = pd.read_csv('RNA_data.csv')

# Your RNADataset class (assumed already defined)
class RNADataset(torch.utils.data.Dataset):
    def __init__(self, df, window_size=20):
        self.window_size = window_size
        self.data = []
        self.labels = []

        seq = df['resname'].values
        coords = df[['x_1', 'y_1', 'z_1']].values
        mapping = {'A': [1, 0, 0, 0], 'U': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
        pad = [0, 0, 0, 0]

        for i in range(len(seq)):
            context = []
            for j in range(i - window_size//2, i + window_size//2 + 1):
                if 0 <= j < len(seq):
                    context.append(mapping.get(seq[j], pad))
                else:
                    context.append(pad)
            self.data.append(torch.tensor(context).T)  # shape: [4, window_size]
            self.labels.append(torch.tensor(coords[i]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx].float(), self.labels[idx].float()

# --- Build dataset from multiple molecules ---

all_data = []

for mol_id, group in df.groupby('mol_id'):
    group = group.sort_values('resid').reset_index(drop=True)
    dataset = RNADataset(group, window_size=5)
    all_data.extend([dataset[i] for i in range(len(dataset))])

# Create a DataLoader
loader = DataLoader(all_data, batch_size=32, shuffle=True)

model = Residue3DPredictor()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
def tm_score_approx(pred, target, d0=1.24):
    """
    pred, target: (B, N, 3) coordinates
    d0: normalization constant
    """
    dists = torch.norm(pred - target, dim=-1)  # (B, N)
    score = 1 / (1 + (dists / d0) ** 2)
    return score.mean()  # average over residues and batch

for epoch in range(2):
    total_loss = 0
    total_tm = 0
    model.train()

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)  # (B, N, features), (B, N, 3)

        optimizer.zero_grad()
        outputs = model(inputs)  # (B, N, 3)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        # Surrogate TM-score
        tm = tm_score_approx(outputs, targets)

        total_loss += loss.item()
        total_tm += tm.item()

    avg_loss = total_loss / len(loader)
    avg_tm = total_tm / len(loader)

    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, TM-score ≈ {avg_tm:.4f}")



In [ ]:
# Save the model
torch.save(model.state_dict(), 'RNA_amodel2.pt')
# Initialize and load the model
model = Residue3DPredictor()
model.load_state_dict(torch.load('RNA_amodel2.pt'))
model.eval()  # Set to evaluation mode


Residue3DPredictor(
  (initial_conv): Conv1d(4, 32, kernel_size=(9,), stride=(1,), padding=(4,))
  (middle_convs): Sequential(
    (0): Sequential(
      (0): Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
    )
    (1): Sequential(
      (0): Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
    )
    (2): Sequential(
      (0): Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
    )
    (3): Sequential(
      (0): Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(32, eps=1e-05,

Evaluate or Visualize
Compare predicted vs. actual 3D coordinates using RMSD, or visualize with matplotlib

In [ ]:
import torch
import random
import pandas as pd

def prepare_sequence(seq, window_size=5):
    mapping = {'A': [1, 0, 0, 0], 'U': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
    pad = [0, 0, 0, 0]
    input_data = []

    for i in range(len(seq)):
        context = []
        for j in range(i - window_size // 2, i + window_size // 2 + 1):
            if 0 <= j < len(seq):
                base = seq[j]
                one_hot = mapping.get(base, random.choice(list(mapping.values())))
                context.append(one_hot)
            else:
                context.append(pad)
        input_data.append(torch.tensor(context).T.float())  # shape: [4, window_size]

    return input_data

# Example sequence
test_seq = ['A', 'U', 'C', 'G', 'A','G','U','A']
window_size = 5
num_runs = 5

# Store predictions for each run
all_run_predictions = []

for run in range(num_runs):
    inputs = prepare_sequence(test_seq, window_size)
    run_predictions = []

    with torch.no_grad():
        for x in inputs:
            x = x.unsqueeze(0)  # Shape: [1, 4, window_size]
            y_pred = model(x)   # Output: [1, 3]
            run_predictions.append(y_pred.squeeze().numpy())  # [3]

    all_run_predictions.append(run_predictions)  # shape: [seq_len, 3] per run

# Transpose to organize by residue
final_predictions = []
for i, residue in enumerate(test_seq):
    row = [residue, i + 1]  # resname, resid (1-based index)
    for run in range(num_runs):
        row.extend(all_run_predictions[run][i])  # x, y, z for each run
    final_predictions.append(row)

# Define column names
coords_columns = [f'{axis}_{run+1}' for run in range(num_runs) for axis in ['x', 'y', 'z']]
columns = ['resname', 'resid'] + coords_columns

# Save to CSV
df = pd.DataFrame(final_predictions, columns=columns)
df.to_csv("predictions.csv", index=False)


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


In [ ]:
#validation

import pandas as pd
import numpy as np

# Load the validation file
val_df = pd.read_csv('./data/validation_labels.csv')  # Assuming this is the file name

# Extract base mol_name and res_index
#val_df['mol_base'] = val_df['ID'].apply(lambda x: '_'.join(x.split('_')[:2]))
#val_df['res_index'] = val_df['ID'].apply(lambda x: int(x.split('_')[-1]))
val_df['mol_base'] = val_df['ID'].apply(lambda x: x.split('_')[0])

# Initialize output list
output_rows = []
global_mol_id = 10001  # Use a distinct ID range for validation set

for mol_base, group in val_df.groupby('mol_base'):
    group = group.sort_values('resid').reset_index(drop=True)

    current_segment = []
    prev_resid = None

    for _, row in group.iterrows():
        # Consider -1e+18 as missing (common in your data)
        missing_coords = any([
            np.isclose(row['x_1'], -1e+18),
            np.isclose(row['y_1'], -1e+18),
            np.isclose(row['z_1'], -1e+18)
        ])
        resid_break = (prev_resid is not None and row['resid'] != prev_resid + 1)

        if (resid_break or missing_coords) and current_segment:
            for r in current_segment:
                r['mol_name'] = mol_base
                r['mol_id'] = global_mol_id
                output_rows.append(r)
            global_mol_id += 1
            current_segment = []

        if not missing_coords:
            current_segment.append(row.to_dict())

        prev_resid = row['resid']

    if current_segment:
        for r in current_segment:
            r['mol_name'] = mol_base
            r['mol_id'] = global_mol_id
            output_rows.append(r)
        global_mol_id += 1

# Create final validation DataFrame
final_df = pd.DataFrame(output_rows)
final_df = final_df[['mol_name', 'mol_id', 'resname', 'resid', 'x_1', 'y_1', 'z_1']]

# Save
final_df.to_csv('RNA_val_data.csv', index=False)

import torch
import torch.nn as nn
import torch.nn.functional as F

class Residue3DPredictor(nn.Module):
    def __init__(self, window_size=20, num_classes=4):  # window_size changed to 10
        super().__init__()
        self.window_size = window_size

        # First convolution layer (4 channels → 32 channels)
        self.initial_conv = nn.Conv1d(num_classes, 32, kernel_size=9, padding=8)  # padding changed to 8

        # 10 convolutional layers (32 → 32), with kernel_size=9, padding=8
        self.middle_convs = nn.Sequential(
            *[nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=9, padding=8),  # padding changed to 8
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(0.1)
            ) for _ in range(10)]
        )

        # Final convolution (32 → 128)
        self.final_conv = nn.Conv1d(32, 128, kernel_size=9, padding=8)  # padding changed to 8

        # Pool to fixed window size
        self.pool = nn.AdaptiveAvgPool1d(output_size=window_size)  # output size changed to 10

        # Fully connected to 3D coordinates
        self.fc = nn.Linear(128 * window_size, 3)  # adjusts automatically to window_size=10

    def forward(self, x):
        x = F.relu(self.initial_conv(x))     # [B, 32, W]
        x = self.middle_convs(x)             # [B, 32, W]
        x = F.relu(self.final_conv(x))       # [B, 128, W]
        x = self.pool(x)                     # [B, 128, 10]
        x = x.view(x.size(0), -1)            # [B, 128 * 10]
        return self.fc(x)


def prepare_sequence(seq, window_size=5):
    mapping = {'A': [1, 0, 0, 0], 'U': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
    pad = [0, 0, 0, 0]
    input_data = []

    for i in range(len(seq)):
        context = []
        for j in range(i - window_size // 2, i + window_size // 2 + 1):
            if 0 <= j < len(seq):
                base = seq[j]
                one_hot = mapping.get(base, random.choice(list(mapping.values())))
                context.append(one_hot)
            else:
                context.append(pad)
        input_data.append(torch.tensor(context).T.float())  # shape: [4, window_size]

    return input_data


# Group by molecule
mol_groups = val_df.groupby('mol_id')

all_inputs = []
all_targets = []
mol_lens = []

criterion = torch.nn.MSELoss()
total_loss = 0.0
mol_losses = []

for mol_id, group in mol_groups:
    group = group.sort_values('resid')
    seq = group['resname'].tolist()
    coords = group[['x_1', 'y_1', 'z_1']].values  # shape: (N, 3)

    # Skip molecules with very few residues if needed
    if len(seq) < 1:
        continue

    # Prepare inputs and targets
    inputs = prepare_sequence(seq, window_size=5)  # list of [4, 5]
    inputs = torch.stack(inputs).to(device)        # [N, 4, 5]
    targets = torch.tensor(coords, dtype=torch.float32).to(device)  # [N, 3]
    all_inputs.append(inputs)
    all_targets.append(targets)
    mol_lens.append(len(seq))
    # Forward pass
    with torch.no_grad():
        preds = model(inputs)  # [N, 3]

    # Compute loss for this molecule
    loss = criterion(preds, targets)
    total_loss += loss.item()
    mol_losses.append((mol_id, loss.item()))

# Average loss across all molecules
avg_loss = total_loss / len(mol_losses)
print(f"Average Loss per molecule: {avg_loss:.4f}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_val = torch.cat(all_inputs, dim=0).to(device)   # shape: [total_residues, 4, 5]
Y_val = torch.cat(all_targets, dim=0).to(device)  # shape: [total_residues, 3]

model = Residue3DPredictor(window_size=5).to(device)
model.load_state_dict(torch.load('RNA_model.pt', map_location=device))
model.eval()

with torch.no_grad():
    Y_pred = model(X_val)  # shape: [total_residues, 3]

from torch.nn import functional as F

def tm_score_approx(pred, target, d0=1.24):
    dists = torch.norm(pred - target, dim=-1)
    score = 1.0 / (1.0 + (dists / d0) ** 2)
    return score.mean()

tm_score = tm_score_approx(Y_pred, Y_val)
print(f"Approximate TM-score: {tm_score.item():.4f}")